### **1주차 프로젝트**

이번 프로젝트에서는 업비트 API를 활용해 암호화폐 데이터를 불러오고, 이를 분석·계산하는 실습을 진행합니다.

과제는 **필수 과제**와 **도전 과제**로 나뉘며, 필수 과제는 반드시 제출해야 하고, 도전 과제는 선택적으로 수행하면 됩니다.

---

**[필수 과제]**
*   문제1. 암호화폐 포트폴리오 분석기
*   문제2. 가격 알림 시스템

**[도전 과제]**
*   문제3. 암호화폐 수익률 계산기

### **[필수 과제] 문제 1: 암호화폐 포트폴리오 분석기**


다음 조건을 만족하는 포트폴리오 분석 프로그램을 작성하시오:

1. 보유 암호화폐와 수량을 딕셔너리로 정의  
2. 각 암호화폐의 현재 가치 계산  
3. 전체 포트폴리오 가치 계산  
4. 각 암호화폐의 포트폴리오 내 비중 계산  

---
|
### 📌 구현 단계
1. **포트폴리오 정의**  
   - 딕셔너리로 보유한 암호화폐와 수량을 저장한다.  
   - 예: `"KRW-BTC": 0.1, "KRW-ETH": 2.5`

```
portfolio = {
    "KRW-BTC":0.1,
    "KRW-EH":3,
    "KRX-XRP":1000
}
```

2. **현재가 조회 함수 작성 (`get_current_prices_api`)**  
   - `requests` 모듈을 사용하여 업비트 API에서 현재가를 조회한다.  
   - 응답을 JSON으로 변환 후, 코인별 현재가(`trade_price`)를 딕셔너리에 담아 반환한다.  

3. **포트폴리오 분석 함수 작성 (`analyze_portfolio`)**  
   (1) **현재가 조회**  
   - 보유 코인 목록을 API에 전달하여 현재가 데이터를 가져온다.  

   (2) **각 암호화폐별 분석**  
   - 현재가 × 보유 수량 → 개별 가치 계산  
   - `portfolio_analysis` 리스트에 코인명, 수량, 현재가, 가치 저장  
   - 동시에 `total_value`(총합)를 누적  

   (3) **비중 계산**  
   - 개별 가치 ÷ 총합 × 100 으로 각 암호화폐의 비중을 추가  
   - `현재가 × 보유 수량`  
   - 예: 비트코인 0.1개 × 50,000,000원 = 5,000,000원  

4. **분석 결과 출력**  
   - 총 포트폴리오 가치를 출력한다.  
   - 표 형태로 코인 / 수량 / 현재가 / 가치 / 비중을 출력한다.   

---

<details>
<summary>💡 구현 힌트 (펼쳐보기)</summary>

- **포트폴리오 정의**  
보유 코인을 딕셔너리 형태로 저장합니다.
  ```python
  portfolio = {
      "KRW-BTC": 0.1,
      "KRW-ETH": 2.5
  }

- **현재가 조회**  
업비트 API를 호출해서 받은 JSON 데이터에서 `trade_price` 값을 꺼내면 됩니다.
  ```python
  for data in prices_data:
      prices[data['market']] = data['trade_price']

- **가치 계산**  
각 코인의 가치는 `현재가 × 수량`으로 구하고, 총합은 반복문에서 누적합니다.
  ```python
  value = price * quantity
  total_value += value

- **비중 계산**  
총합이 구해졌다면, 각 항목에 비중을 추가할 수 있습니다.
  ```python
  item["비중"] = (item["가치"] / total_value) * 100

In [5]:
import requests
from datetime import datetime
import time
import json

# 현재가 조회 함수 작성 (get_current_prices_api)
def get_current_prices_api(ticker_list):

  url = "https://api.upbit.com/v1/ticker"
  headers = {"accept": "application/json"}
  markets_param = ",".join(ticker_list)
  params = {"markets": markets_param}

  response = requests.get(url, headers=headers, params=params)
  prices_data = response.json()

  # 코인별 현재가(trade_price)를 딕셔너리에 담아 반환
  prices = {}
  for data in prices_data:
    prices[data['market']] = data['trade_price']

  return prices


# 포트폴리오 분석 함수 작성 (analyze_portfolio)
def analyze_portfolio(portfolio):

  # 현재가 조회 (보유 코인 목록을 API에 전달하여 현재가 데이터를 가져온다)
  ticker_list = list(portfolio.keys())
  prices = get_current_prices_api(ticker_list)

  # portfolio_analysis 리스트에 코인명, 수량, 현재가, 가치 저장
  # 현재가 × 보유 수량 → 개별 가치
  portfolio_analysis = []
  total_value = 0

  for ticker, quantity in portfolio.items():
    current_price = prices.get(ticker, 0)   # prices에 ticker가 있으면 현재가를 가져오고, 아니면 0을 가져옴
    value = current_price * quantity

    coin_name = ticker.split('-')[1]

    portfolio_analysis.append({
        '코인'   : coin_name,
        '수량'   : quantity,
        '현재가' : current_price,
        '가치'   : value
    })

    total_value += value

  # 비중 계산
  # 개별 가치 ÷ 총합 × 100 으로 각 암호화폐의 비중을 추가
  for item in portfolio_analysis:
    item['비중'] = (item['가치'] / total_value) * 100

  # 표 형태로 코인 / 수량 / 현재가 / 가치 / 비중을 출력
  print('=' * 80)
  print('포트폴리오 분석 결과')
  print('=' * 80)
  print(f"\n총 포트폴리오 가치: {total_value:,.0f}원\n")
  print('-' * 80)
  print(f"{'코인':^10} {'수량':>10} {'현재가':>15} {'가치':>15} {'비중':>12}")
  print('-' * 80)

  for item in portfolio_analysis:
    coin     = item['코인']
    quantity = item['수량']
    price    = item['현재가']
    value    = item['가치']
    ratio    = item['비중']

    print(f"{coin:^10} {quantity:>15.4f} {price:>18,} {value:>18,.0f} {ratio:>11.2f}%")

# 함수 사용 (portfolio 위 예)
portfolio = {
    "KRW-BTC": 0.1,
    "KRW-ETH": 3,
    "KRW-XRP": 1000
}

analyze_portfolio(portfolio)

포트폴리오 분석 결과

총 포트폴리오 가치: 24,022,400원

--------------------------------------------------------------------------------
    코인             수량             현재가              가치           비중
--------------------------------------------------------------------------------
   BTC              0.1000      115,404,000.0         11,540,400       48.04%
   ETH              3.0000        3,449,000.0         10,347,000       43.07%
   XRP           1000.0000            2,135.0          2,135,000        8.89%


### [필수 과제] 문제 2: 가격 알림 시스템

특정 암호화폐의 가격이 목표가에 도달했을 때 알림을 주는 시스템을 만들어보시오:

1. 목표가 설정 (상한가, 하한가)
2. 현재가 모니터링
3. 목표가 도달 시 알림 메시지 출력  

---

### 📌 구현 단계
1. **목표가 설정**  
   - 감시할 코인 티커와 상한가·하한가를 미리 입력한다.
   - 예: `상한가 = 현재가 × 1.05`, `하한가 = 현재가 × 0.95`

2. **현재가 조회 함수 작성 (`get_single_price_api`)**  
   - `requests` 모듈을 사용하여 업비트 API에서 단일 코인의 현재가를 조회한다.
   - 응답을 JSON으로 변환 후, `trade_price` 값을 반환한다.

3. **가격 알림 함수 작성 (`price_alert_system`)**  
   (1) **기본 출력**  
   - 코인명과 상한가·하한가 목표값을 안내 메시지로 출력한다.

   (2) **현재가 조회**  
   - 반복문을 이용해 일정 횟수 동안 현재가를 가져온다.
   - 조회한 시간(`HH:MM:SS`)과 현재가를 함께 출력한다.

   (3) **알림 조건 확인**  
   - 현재가가 상한가 이상이면 상한가 도달 메시지 출력
   - 현재가가 하한가 이하이면 하한가 도달 메시지 출력
   - 두 조건 모두 해당하지 않으면 정상 범위 메시지 출력

4. **모니터링 종료**  
   - 설정한 횟수만큼 확인이 끝나면 모니터링을 종료한다.   

---

<details>
<summary>💡 구현 힌트 (펼쳐보기)</summary>

- **목표가 설정**  
현재가를 기준으로 상한가와 하한가를 계산합니다.
  ```python
  high_target = current_price * 1.05
  low_target  = current_price * 0.95

- **단일 현재가 조회**  
응답 JSON은 리스트이며, 첫 원소 안에 `trade_price`가 들어 있습니다.
  ```python
  data = response.json()[0]
  return data['trade_price']

- **코인명 출력**  
티커 `"KRW-BTC"`에서 `"BTC"`만 추출해 출력할 수 있습니다.
  ```python
  coin_name = ticker.split("-")[1]

- **알림 조건**  
  - 현재가가 상한가 이상이면 상한가 도달 메시지를,
  - 하한가 이하이면 하한가 도달 메시지를 출력합니다.
  ```python
  if current_price >= target_high:
      # 상한가 도달 메시지 출력
  elif current_price <= target_low:
      # 하한가 도달 메시지 출력

- **반복 확인**  
  - 여러 번 가격을 확인해야 하므로 **반복문**을 사용합니다.
  - 각 단계에서 현재 시간과 함께 현재가를 출력하면 더 직관적입니다.
  ```python
  current_time = datetime.now().strftime("%H:%M:%S")
  print(f"[{current_time}] 현재가: {current_price}")

In [10]:
# 현재가 조회 함수 작성 (get_single_price_api)
def get_single_price_api(ticker):
  url = "https://api.upbit.com/v1/ticker"
  headers = {"accept": "application/json"}

  params = {"markets": ticker}

  response = requests.get(url, headers=headers, params=params)
  data = response.json()[0]   # 딕셔너리로 꺼내오기 위해 인덱싱
  return data['trade_price']


# 가격 알림 함수 작성 (price_alert_system)
def price_alert_system(ticker, target_high_price, target_low_price, check_count=30, interval=30):
  coin_name = ticker.split('-')[1]
  print('=' * 80)
  print(f'{coin_name} 가격 알림 시스템 시작')
  print('=' * 80)
  print(f'상한가 목표 : {target_high_price:,.0f}원')
  print(f'하한가 목표 : {target_low_price:,.0f}원')
  print(f'확인 횟수 : {check_count}회')
  print(f'확인 간격 : {interval}초')
  print('-' * 80)

  # check_count만큼 반복하여 확인
  for i in range(check_count):
    current_price = get_single_price_api(ticker)
    current_time = datetime.now().strftime("%H:%M:%S")
    print(f'[{current_time}] 현재가 : {current_price:,.0f}원', end = ' ')

    # 현재가가 상한가 이상이면 상한가 도달 메시지 출력
    if current_price >= target_high_price:
      print(f'상한가 목표 금액 도달! (목표: {target_high_price:,.0f}원)')
    # 현재가가 하한가 이하이면 하한가 도달 메시지 출력
    elif current_price <= target_low_price:
      print(f'하한가 목표 금액 도달! (목표: {target_low_price:,.0f}원)')
    # 두 조건 모두 해당하지 않으면 정상 범위 메시지 출력
    else:
      print('아직 내가 설정한 목표 금액 안에 있습니다.')

    if i <= check_count - 1:
      time.sleep(interval)

  print()
  print('-' * 80)
  print('모니터링 종료')
  print('=' * 80)


# 사용 예시
ticker = "KRW-BTC"
current_price = get_single_price_api(ticker)

target_high_price = current_price * 1.05
target_low_price  = current_price * 0.95

print(f'상한가 : {target_high_price}, 하한가 : {target_low_price}, 현재가 : {current_price}')
price_alert_system(ticker, target_high_price=target_high_price, target_low_price=target_low_price, check_count=30, interval=5)

상한가 : 121351650.0, 하한가 : 109794350.0, 현재가 : 115573000.0
BTC 가격 알림 시스템 시작
상한가 목표 : 121,351,650원
하한가 목표 : 109,794,350원
확인 횟수 : 30회
확인 간격 : 5초
--------------------------------------------------------------------------------
[08:36:35] 현재가 : 115,573,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:36:41] 현재가 : 115,573,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:36:46] 현재가 : 115,573,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:36:52] 현재가 : 115,555,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:36:57] 현재가 : 115,570,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:03] 현재가 : 115,570,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:08] 현재가 : 115,570,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:14] 현재가 : 115,570,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:20] 현재가 : 115,570,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:25] 현재가 : 115,570,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:31] 현재가 : 115,570,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:36] 현재가 : 115,570,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:42] 현재가 : 115,570,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:47] 현재가 : 115,580,000원 아직 내가 설정한 목표 금액 안에 있습니다.
[08:37:53



### [도전 과제] 문제 3: 암호화폐 수익률 계산기

과거 특정 시점에 투자했다면 현재 수익률이 얼마인지 계산하는 프로그램을 작성하시오:

1. 투자 시점과 투자 금액 설정
2. 해당 시점의 가격과 현재 가격 비교
3. 수익률과 수익금 계산

---

### 📌 구현 단계
1. **투자 조건 설정**  
   - 투자할 코인 티커, 투자 시점(며칠 전), 투자 금액을 입력한다.
   - 예: 비트코인, 30일 전, 100만원

2. **과거 데이터 조회 함수 작성 (`get_historical_data_api`)**  
   - 업비트 `candles/days` API를 사용해 지정한 기간의 일봉 데이터를 가져온다.
   - 응답 데이터에서 투자 시점의 종가(`trade_price`)를 추출한다.

3. **현재가 조회 함수 작성 (`get_single_price_api`)**  
   - 단일 티커의 현재 가격을 조회한다.
   - JSON 응답에서 trade_price를 반환한다.  

4. **투자 수익률 계산 함수 작성 (`calculate_investment_return`)**  
   (1) **투자 시점 가격 확인**  
   - 투자 시점 가격으로 투자 금액 ÷ 가격 = 구매 수량을 계산한다.

   (2) **현재 가치 계산**  
   - 구매 수량 × 현재가 = 현재 가치

   (3) **손익 및 수익률 계산**  
   - 손익 = 현재 가치 – 투자 금액
   - 수익률 = (손익 ÷ 투자 금액) × 100  

5. **결과 출력**  
   - 투자 일자, 투자 금액, 투자 시점 가격, 구매 수량, 현재 가격, 현재 가치, 손익, 수익률을 순서대로 출력한다.


---

<details>
<summary>💡 구현 힌트 (펼쳐보기)</summary>

- **과거 데이터 불러오기**  
  - 업비트 일봉 API(`candles/days`)를 사용하면 과거 일자별 가격 데이터를 조회할 수 있습니다.
  - 응답 리스트에서 `'trade_price'`(종가)를 이용하면 투자 시점 가격을 알 수 있습니다.
  ```python
  url = "https://api.upbit.com/v1/candles/days"
  params = {"market": ticker, "count": days_count}
  response = requests.get(url, params=params)
  candle_data = response.json()

- **투자 시점 가격 구하기**  
  - `days_ago`일 전의 가격을 꺼내 투자 금액으로 몇 개를 살 수 있는지 계산합니다.
  - (힌트: `구매 수량 = 투자 금액 ÷ 투자 시점 가격`)  

- **현재 가격 확인하기**  
단일 현재가 조회 API를 활용해 `trade_price`를 가져옵니다.
  ```python
  data = response.json()[0]
  return data['trade_price']

- **수익률 계산하기**  
  - 현재 가치 = `보유 수량 × 현재가`
  - 손익 = `현재 가치 - 투자금액`
  - 수익률 = `(손익 ÷ 투자금액) × 100`

- **결과 출력**  
  - 투자 일자, 투자 금액, 투자 시점 가격, 구매 수량, 현재 가치, 손익, 수익률을 차례대로 출력하세요.
  - 날짜는 `datetime`을 사용해 `"YYYY-MM-DD"` 형식으로 바꿔줄 수 있습니다.
  ```python
  investment_date.strftime('%Y-%m-%d')


In [13]:
import requests
import time
import json
from datetime import datetime, timedelta

# 과거 데이터 조회 함수 작성 (get_historical_data_api)
def get_historical_data_api(ticker, days_count):

  url = "https://api.upbit.com/v1/candles/days"
  headers = {"accept": "application/json"}

  params = {
      'market' : ticker,
      'count'  : days_count
  }

  response = requests.get(url, headers=headers, params=params)
  candle_data = response.json()

  candle_list = []
  for i in range(days_count):
    candle_list.append(candle_data[i])

  return candle_list


# 현재가 조회 함수 작성 (get_single_price_api)
def get_single_price_api(ticker):
  url = "https://api.upbit.com/v1/ticker"
  headers = {"accept": "application/json"}

  params = {"markets": ticker}

  response = requests.get(url, headers=headers, params=params)
  data = response.json()[0]
  return int(data['trade_price'])


# 투자 수익률 계산 함수 작성 (calculate_investment_return)
def calculate_investment_return(ticker, days_count, invest_amount):

  candle_data = get_historical_data_api(ticker, days_count)

  # 현재 가격 확인
  current_price = get_single_price_api(ticker)

  for i in range(days_count-1):
    # 투자 시점 가격 확인
    invest_date = (datetime.now() - timedelta(days=i+1)).strftime('%Y-%m-%d')
    invest_price = int(candle_data[i+1]['trade_price'])
    invest_quantity = invest_amount / invest_price

    # 현재 가치 계산
    current_value = int(invest_quantity * current_price)

    # 손익 및 수익률 계산
    profit = current_value - invest_amount
    profit_ratio = (profit / invest_amount) * 100

    print(f"{invest_date:^10} {invest_amount:>15,} {invest_price:>18,} {invest_quantity:>15.5f} {current_price:>20,} {current_value:>19,} {profit:>15,} {profit_ratio:>12.2f}%")


# 사용 예시
ticker = 'KRW-BTC'
days_count = 30
invest_amount = 1000000

print('-' * 200)
print('암호화폐 수익률 계산 분석 결과')
print('-' * 200)
print(f"{'투자 일자':^10} {'투자 금액':>7} {'투자 시점 가격':>13} {'구매 수량':>11} {'현재 가격':>15} {'현재 가치':>15} {'손익':>13} {'수익률':>10}")
print('-' * 200)
calculate_investment_return(ticker, days_count=days_count, invest_amount=invest_amount)


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
암호화폐 수익률 계산 분석 결과
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  투자 일자      투자 금액      투자 시점 가격       구매 수량           현재 가격           현재 가치            손익        수익률
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
2026-04-24       1,000,000        115,463,000         0.00866          115,883,000           1,003,637           3,637         0.36%
2026-04-23       1,000,000        116,429,000         0.00859          115,883,000             995,310          -4,690        -0.47%
2026-04-22 


## [고급자용 추가 문제]

기존 프로젝트를 마친 수강생을 위한 확장 문제입니다.
아래 문제는 **데이터 분석 + 시각화 + 함수 분리**를 함께 연습하는 것이 목표입니다.

**추천 순서**
1. 문제 3을 먼저 해결한다.
2. 문제 4에서 여러 코인을 비교한다.
3. 문제 5에서 시계열 분석과 시각화를 적용한다.



### [고급자용] 문제 4: 여러 암호화폐 수익률 비교 분석기

다음 조건을 만족하는 프로그램을 작성하시오.

1. 비교할 암호화폐 티커 목록을 리스트로 정의한다.
2. 각 코인의 **N일 전 종가**와 **현재가**를 조회한다.
3. 각 코인별 수익률을 계산한다.
4. 수익률 기준으로 내림차순 정렬하여 출력한다.
5. 가장 많이 상승한 코인과 가장 많이 하락한 코인을 따로 출력한다.

---

### 구현 단계
1. `tickers = ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL"]` 형태로 비교 대상 정의
2. `get_historical_close_api(ticker, days_ago)` 함수 작성
3. 현재가 조회 함수와 결합하여 코인별 수익률 계산
4. 결과를 리스트 또는 딕셔너리로 저장
5. 수익률 기준 정렬 후 보기 좋게 출력

---

<details>
<summary>구현 힌트</summary>

- 과거 종가는 `candles/days` API 응답의 `trade_price`를 활용할 수 있습니다.
- 수익률 공식
```python
return_rate = ((current_price - past_price) / past_price) * 100
```
- 정렬 예시
```python
results.sort(key=lambda x: x["수익률"], reverse=True)
```
- 출력 컬럼 예시
  - 코인명
  - N일 전 가격
  - 현재가
  - 수익률

</details>


In [24]:
tickers = ["KRW-BTC", "KRW-ETH", "KRW-XRP", "KRW-SOL"]
days_ago = 20

# 종가 계산 함수 작성 (get_historical_close_api)
def get_historical_close_api(ticker, days_ago):

  url = "https://api.upbit.com/v1/candles/days"
  headers = {"accept": "application/json"}

  params = {
      'market' : ticker,
      'count'  : days_ago
  }

  response = requests.get(url, headers=headers, params=params)
  candle_data = response.json()[-1]
  return int(candle_data['trade_price'])

# 코인별 수익률 계산
results = []

for ticker in tickers:
  past_price    = get_historical_close_api(ticker, days_ago)
  current_price = get_single_price_api(ticker)

  # 수익률
  return_rate = ((current_price - past_price) / past_price) * 100

  # 결과를 리스트 또는 딕셔너리로 저장
  coin_name = ticker.split('-')[1]
  results.append({
      '코인명'      : coin_name,
      'N일 전 가격' : past_price,
      '현재가'      : current_price,
      '수익률'      : return_rate
  })

# 수익률 기준 정렬
results.sort(key=lambda x: x['수익률'], reverse=True)

print('-' * 80)
print(f"{'코인명':^10} {'N일 전 가격':>11} {'현재가':>13} {'수익률':>10}")
print('-' * 80)

for i in range(len(results)):
  coin_name     = results[i]['코인명']
  past_price    = results[i]['N일 전 가격']
  current_price = results[i]['현재가']
  return_rate   = results[i]['수익률']
  print(f"{coin_name:^10} {past_price:>18,} {current_price:>18,} {return_rate:>10.2f}%")


--------------------------------------------------------------------------------
   코인명         N일 전 가격           현재가        수익률
--------------------------------------------------------------------------------
   BTC            103,923,000        115,896,000      11.52%
   ETH              3,182,000          3,456,000       8.61%
   XRP                  1,996              2,135       6.96%
   SOL                120,800            129,200       6.95%
